In [0]:
from pyspark.sql.functions import (
    col, from_json, trim, concat_ws, current_timestamp,
    split, explode_outer, length, expr
)
from pyspark.sql.types import (
    StructType, StructField, StringType, ArrayType, MapType
)

# 1. CATALOG & SCHEMA SETUP
CATALOG = "pipeline_signal"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")

def save_to_silver(df, table_name):
    full_table = f"{CATALOG}.{SILVER_SCHEMA}.{table_name}"
    df.write \
      .format("delta") \
      .mode("overwrite") \
      .option("overwriteSchema", "true") \
      .option("delta.enableChangeDataFeed", "true") \
      .saveAsTable(full_table)
    print(f"✅ Created Silver Table: `{full_table}` (Total Rows: {df.count()})")


# 2. SILVER: dbt Models & Lineage
print("--- [1/4] Processing Silver: dbt Models ---")
dbt_node_schema = StructType([
    StructField("name", StringType(), True),
    StructField("resource_type", StringType(), True),
    StructField("database", StringType(), True),
    StructField("schema", StringType(), True),
    StructField("package_name", StringType(), True),
    StructField("description", StringType(), True),
    StructField("depends_on", StructType([
        StructField("nodes", ArrayType(StringType()), True)
    ]), True)
])

df_bronze_manifest = spark.read.table(f"{CATALOG}.{BRONZE_SCHEMA}.dbt_manifest")
df_silver_dbt = df_bronze_manifest \
    .withColumn("parsed_node", from_json(col("raw_node"), dbt_node_schema)) \
    .select(
        col("node_id"),
        col("parsed_node.name").alias("model_name"),
        col("parsed_node.resource_type").alias("resource_type"),
        col("parsed_node.database").alias("database"),
        col("parsed_node.schema").alias("schema"),
        col("parsed_node.package_name").alias("package_name"),
        col("parsed_node.description").alias("description"),
        col("raw_code"),
        col("parsed_node.depends_on.nodes").alias("upstream_depends_on"),
        col("ingested_at")
    ) \
    .withColumn("processed_at", current_timestamp())

save_to_silver(df_silver_dbt, "silver_dbt_models")


# 3. SILVER: DataHub Lineage
print("\n--- [2/4] Processing Silver: DataHub Lineage ---")
mce_schema = StructType([
    StructField("proposedSnapshot", StructType([
        StructField("com.linkedin.pegasus2avro.metadata.snapshot.DatasetSnapshot", StructType([
            StructField("urn", StringType(), True)
        ]), True)
    ]), True)
])

df_bronze_mce = spark.read.table(f"{CATALOG}.{BRONZE_SCHEMA}.metadata_mce")
df_silver_mce = df_bronze_mce \
    .withColumn("parsed_mce", from_json(col("raw_event"), mce_schema)) \
    .select(
        col("parsed_mce.proposedSnapshot.`com.linkedin.pegasus2avro.metadata.snapshot.DatasetSnapshot`.urn").alias("dataset_urn"),
        col("raw_event"),
        col("ingested_at")
    ) \
    .withColumn("processed_at", current_timestamp())

save_to_silver(df_silver_mce, "silver_datahub_lineage")


# 4. SILVER: GitHub Issues
print("\n--- [3/4] Processing Silver: GitHub Issues ---")
df_bronze_issues = spark.read.table(f"{CATALOG}.{BRONZE_SCHEMA}.github_issues")
df_silver_issues = df_bronze_issues \
    .select(
        col("issue_id"),
        col("number").cast("integer").alias("issue_number"),
        trim(col("title")).alias("title"),
        trim(col("body")).alias("body"),
        col("state"),
        col("html_url"),
        concat_ws(" - ", trim(col("title")), trim(col("body"))).alias("search_text"),
        col("ingested_at")
    ) \
    .withColumn("processed_at", current_timestamp())

save_to_silver(df_silver_issues, "silver_github_issues")


# 5. SILVER: Chunked Documentation (Pure PySpark - No Pandas required)
print("\n--- [4/4] Processing Silver: Chunked Documentation ---")
df_bronze_docs = spark.read.table(f"{CATALOG}.{BRONZE_SCHEMA}.documentation")

df_silver_docs = df_bronze_docs \
    .withColumn("chunk_text", explode_outer(split(col("content"), "\n\n"))) \
    .select(
        col("doc_id"),
        col("path"),
        col("title"),
        col("category"),
        trim(col("chunk_text")).alias("chunk_text"),
        col("html_url"),
        col("ingested_at")
    ) \
    .filter(col("chunk_text").isNotNull() & (length(col("chunk_text")) > 20)) \
    .withColumn("processed_at", current_timestamp())

save_to_silver(df_silver_docs, "silver_documentation_chunked")

print("\n🎉 ALL 4 SILVER TABLES CREATED SUCCESSFULLY!")